In [1]:
if True:
    from google.colab import drive
    from pathlib import Path

    drive.mount("/content/drive")

    WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
    R2DREAMER_DIR = WORKDIR / "r2dreamer"
    WORKDIR.mkdir(parents=True, exist_ok=True)

    print("Workdir:", WORKDIR)
    print("R2Dreamer:", R2DREAMER_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workdir: /content/drive/MyDrive/Research/SSM_World_Models
R2Dreamer: /content/drive/MyDrive/Research/SSM_World_Models/r2dreamer


In [11]:
if True:
    import importlib
    import os
    import shutil
    import subprocess
    import sys
    import time

    from IPython.display import clear_output


    def run(cmd, cwd=None):
        print("Running:", " ".join(map(str, cmd)))
        subprocess.run(cmd, cwd=cwd, check=True)


    repo = "https://github.com/hanswalker/r2dreamer.git"
    branch = "main"

    if R2DREAMER_DIR.exists():
        run(["git", "remote", "set-url", "origin", repo], cwd=R2DREAMER_DIR)
        run(["git", "fetch", "origin"], cwd=R2DREAMER_DIR)
        run(["git", "checkout", "-B", branch, f"origin/{branch}"], cwd=R2DREAMER_DIR)
        run(["git", "reset", "--hard", f"origin/{branch}"], cwd=R2DREAMER_DIR)
    else:
        run(["git", "clone", "--branch", branch, repo, str(R2DREAMER_DIR)])

    os.chdir(R2DREAMER_DIR)
    if str(R2DREAMER_DIR) not in sys.path:
        sys.path.insert(0, str(R2DREAMER_DIR))

    from dmc_expert_colab import setup_colab

    TDMPC2_DIR, DATA_DIR, Mamba3 = setup_colab(WORKDIR, R2DREAMER_DIR)

    import dmc_expert_collection
    import dmc_expert_training
    import dmc_expert_evaluation

    importlib.reload(dmc_expert_collection)
    importlib.reload(dmc_expert_training)
    importlib.reload(dmc_expert_evaluation)

    from dmc_expert_collection import (
        load_collection_config,
        make_collect_config,
        read_progress,
        start_collector,
    )
    from dmc_expert_training import read_metric_summary, start_training
    from dmc_expert_evaluation import eval_run

    SCENARIOS = [
        {"name": "cartpole", "task": "cartpole/swingup"},
        {"name": "walker", "task": "walker/walk"},
        {"name": "humanoid", "task": "humanoid/run"},
    ]

    MODELS = [
        {"name": "gru", "config": "offline_dmc_expert_gru"},
        {"name": "mamba3", "config": "offline_dmc_expert_mamba3"},
    ]

    DATASETS = {
        "cartpole": DATA_DIR / "cartpole_swingup" / "cartpole_swingup.zarr",
        "walker": DATA_DIR / "walker_walk" / "walker_walk.zarr",
        "humanoid": DATA_DIR / "humanoid_run" / "humanoid_run.zarr",
    }


    def short(path):
        return str(Path(path).relative_to(WORKDIR))

    print("TD-MPC2:", TDMPC2_DIR)
    print("Data:", DATA_DIR)
    print("Datasets:")
    for name, path in DATASETS.items():
        print(f"  {name:<10} {path}")
    print("Ready.")


Running: git remote set-url origin https://github.com/hanswalker/r2dreamer.git
Running: git fetch origin
Running: git checkout -B main origin/main
Running: git reset --hard origin/main

== Update TD-MPC2 ==

== Pull TD-MPC2 ==

== Install Python packages ==

== Remove optional TileLang kernels ==

Ready
R2Dreamer: /content/drive/MyDrive/Research/SSM_World_Models/r2dreamer
TD-MPC2: /content/drive/MyDrive/Research/SSM_World_Models/tdmpc2
Data: /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert
Mamba3: <class 'mamba_ssm.modules.mamba3.Mamba3'>
CUDA: True
TD-MPC2: /content/drive/MyDrive/Research/SSM_World_Models/tdmpc2
Data: /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert
Datasets:
  cartpole   /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/cartpole_swingup/cartpole_swingup.zarr
  walker     /content/drive/MyDrive/Research/SSM_World_Models/data/dmc_expert/walker_walk/walker_walk.zarr
  humanoid   /content/drive/MyDrive/Research/SSM_World_Mod

In [3]:
if True:
    cfg = load_collection_config(
        R2DREAMER_DIR / "configs" / "dmc_expert_collection.yaml",
        tdmpc2_dir=TDMPC2_DIR,
        data_dir=DATA_DIR,
    )

    collector = R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py"
    resume = bool(cfg["resume"])
    target = int(cfg["num_episodes"])
    refresh = int(cfg["refresh_seconds"])


    jobs = []
    for scenario in SCENARIOS:
        out_dir = DATASETS[scenario["name"]].parent

        if out_dir.exists() and not resume:
            shutil.rmtree(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        start_episodes, _, _ = read_progress(out_dir, scenario["task"])
        config_path = out_dir / "collect_config.yaml"
        log_path = out_dir / "collector.log"
        make_collect_config(cfg, scenario["task"], out_dir, config_path)
        proc, log_file = start_collector(collector, config_path, log_path)

        jobs.append({
            "scenario": scenario,
            "out_dir": out_dir,
            "log": log_path,
            "proc": proc,
            "log_file": log_file,
            "start_episodes": start_episodes,
        })

    while True:
        clear_output(wait=True)
        print(f"Data collection: {len(jobs)} tasks x {target} target episodes, resume={resume}")
        print(f"{'task':<18} {'status':<10} {'start':>8} {'episodes':>11} {'rows':>10} log")

        done = True
        for job in jobs:
            code = job["proc"].poll()
            status = "running" if code is None else ("done" if code == 0 else f"failed {code}")

            episodes, rows, _ = read_progress(job["out_dir"], job["scenario"]["task"])
            print(
                f"{job['scenario']['name']:<18} {status:<10} "
                f"{job['start_episodes']:>8} {episodes:>4}/{target:<6} {rows:>10} "
                f"{short(job['log'])}"
            )
            done = done and code is not None

        if done:
            break
        time.sleep(refresh)

    failed = []
    for job in jobs:
        job["log_file"].close()
        if job["proc"].returncode != 0:
            failed.append(job)

    if failed:
        for job in failed:
            print()
            print(f"--- {job['scenario']['name']} last log lines ---")
            print(chr(10).join(job["log"].read_text(errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"{len(failed)} collectors failed")

    print("Datasets:")
    for job in jobs:
        print(f"{job['scenario']['name']:<18} -> {short(job['out_dir'])}")

    print("Collection finished.")


Data collection: 3 tasks x 10000 target episodes, resume=True
task               status        start    episodes       rows log
cartpole           done           4000 10000/10000     5000000 data/dmc_expert/cartpole_swingup/collector.log
walker             done           4000 10000/10000     5000000 data/dmc_expert/walker_walk/collector.log
humanoid           done           4000 10000/10000     5000000 data/dmc_expert/humanoid_run/collector.log
Datasets:
cartpole           -> data/dmc_expert/cartpole_swingup
walker             -> data/dmc_expert/walker_walk
humanoid           -> data/dmc_expert/humanoid_run
Collection finished.


In [ ]:
if True:
    run_tag = time.strftime("%Y%m%d_%H%M%S")

    RUNS = []
    for scenario in SCENARIOS:
        for model in MODELS:
            RUNS.append({
                "name": f"{scenario['name']}_{model['name']}",
                "config": model["config"],
                "data": DATASETS[scenario["name"]],
                "task": "dmc_" + scenario["task"].replace("/", "_"),
                "logdir": WORKDIR / "runs" / f"offline_{scenario['name']}_{model['name']}_{run_tag}",
            })
    print("Training datasets:")
    for run in RUNS:
        print(f"{run['name']:<18} {short(run['data'])}")

    jobs = []
    for run in RUNS:
        proc, log_file, stdout_log = start_training(
            r2dreamer_dir=R2DREAMER_DIR,
            config=run["config"],
            data=run["data"],
            task=run["task"],
            logdir=run["logdir"],
            resume=False,
        )
        jobs.append({"run": run, "proc": proc, "log_file": log_file, "stdout": stdout_log})

    while True:
        clear_output(wait=True)
        print("Offline training: GRU vs Mamba3")
        print(f"{'run':<18} {'status':<10} {'step':>8} {'loss':>10} {'eval':>10} {'ckpt':<9} log")

        done = True
        for job in jobs:
            run = job["run"]
            code = job["proc"].poll()
            status = "running" if code is None else ("done" if code == 0 else f"failed {code}")
            step, loss, score = read_metric_summary(run["logdir"])
            ckpt = "latest" if (run["logdir"] / "latest.pt").exists() else "-"
            ckpt = "best" if (run["logdir"] / "best.pt").exists() else ckpt
            print(
                f"{run['name']:<18} {status:<10} {step:>8} "
                f"{loss:>10} {score:>10} {ckpt:<9} {short(job['stdout'])}"
            )
            done = done and code is not None

        if done:
            break
        time.sleep(10)

    failed = []
    for job in jobs:
        job["log_file"].close()
        if job["proc"].returncode != 0:
            failed.append(job)

    if failed:
        for job in failed:
            print()
            print(f"--- {job['run']['name']} last log lines ---")
            print(chr(10).join(job["stdout"].read_text(errors="replace").splitlines()[-80:]))
        raise RuntimeError(f"{len(failed)} training runs failed")

    print("Training finished.")


Offline training: GRU vs Mamba3
run                status         step       loss       eval ckpt      log
cartpole_gru       running           -          -          - -         runs/offline_cartpole_gru_20260624_002911/notebook_stdout.log
cartpole_mamba3    running           -          -          - -         runs/offline_cartpole_mamba3_20260624_002911/notebook_stdout.log
walker_gru         running           -          -          - -         runs/offline_walker_gru_20260624_002911/notebook_stdout.log
walker_mamba3      running           -          -          - -         runs/offline_walker_mamba3_20260624_002911/notebook_stdout.log
humanoid_gru       running           -          -          - -         runs/offline_humanoid_gru_20260624_002911/notebook_stdout.log
humanoid_mamba3    running           -          -          - -         runs/offline_humanoid_mamba3_20260624_002911/notebook_stdout.log


In [ ]:
if True:
    def newest_logdir(scenario_name, model_name):
        prefix = f"offline_{scenario_name}_{model_name}_"
        candidates = [path for path in (WORKDIR / "runs").glob(prefix + "*") if path.is_dir()]
        if not candidates:
            raise FileNotFoundError(f"No run folder found for {scenario_name}_{model_name}")
        return max(candidates, key=lambda path: path.stat().st_mtime)


    RUNS = []
    for scenario in SCENARIOS:
        for model in MODELS:
            RUNS.append({
                "name": f"{scenario['name']}_{model['name']}",
                "config": model["config"],
                "data": DATASETS[scenario["name"]],
                "task": "dmc_" + scenario["task"].replace("/", "_"),
                "logdir": newest_logdir(scenario["name"], model["name"]),
            })

    results = []
    for run in RUNS:
        print(f"Evaluating {run['name']}...")
        result = eval_run(
            r2dreamer_dir=R2DREAMER_DIR,
            config=run["config"],
            data=run["data"],
            task=run["task"],
            logdir=run["logdir"],
            episodes=5,
            checkpoint="best.pt",
        )
        results.append((run, result))

    print()
    print("Eval summary")
    for run, item in results:
        print(
            f"{run['name']:<18} fresh={item['fresh_eval_score']} "
            f"best_logged={item['logged_best_eval']} ckpt={item['checkpoint']}"
        )

    results
